# County Rainfall and Planting Advisory

**Analyst:** Timothy Kerre | **Cohort:** 1milliondevs 2026 | **Assignment:** PLP Data Analytics A2

**Scenario:** A farmer-support NGO sends SMS advice on planting windows in five counties. This notebook cleans a hand-typed Machakos rainfall record, analyses monthly rainfall for all five counties with NumPy, saves a summary CSV, and ends with a short planting advisory and data-quality notes.

## Dataset

The data below is used exactly as provided. Rows are counties and columns are January to December (rainfall in mm). The Machakos row arrived as raw text and is cleaned in Task 1.

In [1]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

## Task 1: Clean the Machakos data

**Approach:** `clean_machakos()` tries `int()` on every raw value inside a `try` block and catches only `ValueError`, which is what `int()` raises for text such as `"N/A"` or an empty string. Surrounding spaces are not an error, because `int(" 20 ")` returns `20`, so that reading is kept. The invalid positions are recorded, and each is replaced with the rounded average of the valid values. The imputed months are then printed.

In [2]:
def clean_machakos(raw_values):
    """Convert raw Machakos rainfall text to integers, imputing invalid entries.

    Each value is converted with int() inside try/except ValueError. Values that
    cannot be converted (for example "N/A" or an empty string) are replaced with
    the rounded average of the valid values.

    Args:
        raw_values (list): Raw rainfall readings as strings, January to December.

    Returns:
        tuple: (cleaned, imputed_idx, fill_value) where cleaned is a list of
            integers, imputed_idx is a list of the month positions (0-11) that
            were imputed, and fill_value is the integer used to fill them.
    """
    converted = []
    imputed_idx = []
    for i, value in enumerate(raw_values):
        try:
            converted.append(int(value))
        except ValueError:
            converted.append(None)
            imputed_idx.append(i)

    valid = [v for v in converted if v is not None]
    fill_value = round(sum(valid) / len(valid))
    cleaned = [fill_value if v is None else v for v in converted]
    return cleaned, imputed_idx, fill_value


machakos_clean, imputed_idx, fill_value = clean_machakos(machakos_raw)

print("Cleaned Machakos row:", machakos_clean)
print(f"Average of the {len(machakos_raw) - len(imputed_idx)} valid readings, rounded: {fill_value} mm")
for i in imputed_idx:
    print(f"Imputed: {months[i]} (raw value {machakos_raw[i]!r}) -> {fill_value} mm")

Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Average of the 10 valid readings, rounded: 68 mm
Imputed: April (raw value 'N/A') -> 68 mm
Imputed: September (raw value '') -> 68 mm


## Task 2: Combine the five counties into one NumPy array

**Approach:** The four clean rows and the cleaned Machakos row are stacked into a single `5 x 12` array with `np.array()`. Its shape and dtype are printed to confirm the structure and that all values are integers.

In [3]:
rain = np.array(clean_rows + [machakos_clean])

print("Shape:", rain.shape)
print("Dtype:", rain.dtype)
print()
print(rain)

Shape: (5, 12)
Dtype: int64

[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## Task 3: Annual totals and monthly averages

**Approach:** Annual totals sum across the months for each county (`axis=1`). The average rainfall per month across all counties takes the mean down the counties (`axis=0`). Both calculations are vectorised, with no Python loops. A loop is used only to print the results.

In [4]:
annual_totals = rain.sum(axis=1)
monthly_avg = rain.mean(axis=0)

print("Annual rainfall per county:")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<9} {total:>6,} mm")

print("\nAverage rainfall per month across all five counties:")
for month, avg in zip(months, monthly_avg):
    print(f"  {month:<10} {avg:>6.1f} mm")

Annual rainfall per county:
  Nairobi    1,045 mm
  Nakuru       916 mm
  Kisumu     1,371 mm
  Mombasa    1,075 mm
  Machakos     811 mm

Average rainfall per month across all five counties:
  January      47.0 mm
  February     47.6 mm
  March        91.0 mm
  April       164.4 mm
  May         171.2 mm
  June         67.6 mm
  July         56.6 mm
  August       58.2 mm
  September    64.2 mm
  October      67.8 mm
  November    127.4 mm
  December     80.6 mm


## Task 4: Wettest month for each county

**Approach:** `np.argmax(rain, axis=1)` returns the column index of the maximum in each row. Those indices are used to look up month names in a NumPy array of names, and `zip()` pairs each county with its result for printing. `rain.max(axis=1)` supplies the rainfall amount.

In [5]:
month_names = np.array(months)

wettest_idx = np.argmax(rain, axis=1)
wettest_month = month_names[wettest_idx]
wettest_mm = rain.max(axis=1)

for county, month, mm in zip(counties, wettest_month, wettest_mm):
    print(f"{county:<9} wettest month: {month:<9} ({mm} mm)")

Nairobi   wettest month: April     (219 mm)
Nakuru    wettest month: April     (145 mm)
Kisumu    wettest month: April     (240 mm)
Mombasa   wettest month: May       (240 mm)
Machakos  wettest month: November  (182 mm)


## Task 5: Dry months (below 50 mm)

**Approach:** The comparison `rain < 50` creates a boolean mask the same shape as the data. Summing the mask along `axis=1` counts the `True` values in each row, because `True` counts as 1. The mask also identifies which months were dry.

In [6]:
dry_mask = rain < 50
dry_months = dry_mask.sum(axis=1)

for i, (county, count) in enumerate(zip(counties, dry_months)):
    names = ", ".join(month_names[dry_mask[i]]) or "none"
    print(f"{county:<9} dry months: {count}  ({names})")

Nairobi   dry months: 4  (June, July, August, September)
Nakuru    dry months: 3  (January, February, December)
Kisumu    dry months: 0  (none)
Mombasa   dry months: 2  (January, February)
Machakos  dry months: 6  (January, February, June, July, August, October)


## Task 6: Planting window (two consecutive months with the highest rainfall)

**Approach:** The shifted-array technique `rain[:, :-1] + rain[:, 1:]` adds each month to the following month, giving 11 two-month totals per county (January-February up to November-December). `argmax(axis=1)` finds the best pair for each county. The window starts at that index and ends at the next month. Windows do not wrap from December to January, because the data covers one calendar year.

In [7]:
pair_totals = rain[:, :-1] + rain[:, 1:]
window_start = np.argmax(pair_totals, axis=1)
window_mm = pair_totals.max(axis=1)

planting_window = np.char.add(
    np.char.add(month_names[window_start], "-"),
    month_names[window_start + 1],
)

for county, window, mm in zip(counties, planting_window, window_mm):
    print(f"{county:<9} planting window: {window:<15} ({mm} mm combined)")

Nairobi   planting window: April-May       (395 mm combined)
Nakuru    planting window: April-May       (265 mm combined)
Kisumu    planting window: April-May       (420 mm combined)
Mombasa   planting window: April-May       (390 mm combined)
Machakos  planting window: November-December (278 mm combined)


## Task 7: Write `rainfall_summary.csv`

**Approach:** The `csv` module writes one header row and one row per county with the columns `county`, `annual_total_mm`, `wettest_month`, `dry_months` and `planting_window`. NumPy values are converted to plain Python types before writing. The file is then read back and printed to confirm that it has exactly five data rows.

In [8]:
header = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for county, total, month, dry, window in zip(
        counties, annual_totals, wettest_month, dry_months, planting_window
    ):
        writer.writerow([county, int(total), str(month), int(dry), str(window)])

with open("rainfall_summary.csv", newline="") as f:
    rows = list(csv.reader(f))

print(f"Rows written (excluding header): {len(rows) - 1}\n")
for row in rows:
    print(", ".join(row))

Rows written (excluding header): 5

county, annual_total_mm, wettest_month, dry_months, planting_window
Nairobi, 1045, April, 4, April-May
Nakuru, 916, April, 3, April-May
Kisumu, 1371, April, 0, April-May
Mombasa, 1075, May, 2, April-May
Machakos, 811, November, 6, November-December


## Planting Advisory

Nairobi, Nakuru, Kisumu and Mombasa all have their best planting window in April-May (combined rainfall of 395, 265, 420 and 390 mm), so one SMS in late March can prompt land preparation across these four counties. Machakos is the exception, with its wettest window in November-December (278 mm), so farmers there should plan for the short rains rather than April-May. Kisumu is the wettest county at 1,371 mm a year with no dry months, whereas Machakos has six months below 50 mm, so its messages should stress drought-tolerant crops and water conservation. Nairobi has four dry months (June to September), so farmers there should not plan to plant in that period without irrigation. Two Machakos readings (April and September) were missing and were replaced with an estimate of 68 mm, so the Machakos advice should be confirmed against station records before it is sent. These windows are based on one year of monthly totals and should be read as a guide rather than a forecast.

## Data Quality Notes

- **Machakos imputation:** Two of the 12 Machakos readings were invalid: April (`"N/A"`) and September (an empty string). Both were replaced with 68 mm, the rounded average of the ten valid readings (67.5 mm). This estimate flattens any real seasonal pattern.
- **Spaces in the raw data:** `" 20 "` (June) was kept as 20 mm, because `int()` accepts surrounding spaces. This was a formatting problem, not a missing value.
- **Effect on the results:** The Machakos annual total (811 mm) includes 136 mm of estimated rainfall. The Machakos planting window could change if the true April reading was above 138 mm, because April-May would then exceed the November-December total of 278 mm. The wettest month (November) and its dry-month count are less sensitive, because 68 mm is above the 50 mm threshold.
- **Other counties:** The four other rows were used as supplied and were not checked against source records.
- **Scope:** The data covers a single year, so these results show one year's pattern and not a long-term climate average. Planting windows do not wrap from December into January.